# 🏆 Customer Lifetime Value (CLTV) Prediction System
## Phase 5: Feature Engineering
**Project**: Fortune 500 E-Commerce CLTV Prediction  
**Author**: Nisarga N  
**Date**: June 2026  
**Dataset**: UCI Online Retail II  
**Objective**: Build structured customer profiles using observation data and construct predictive targets to prevent data leakage.


In [ ]:
import sys
import warnings
from pathlib import Path
import pandas as pd
import numpy as np

# Add src to system path
sys.path.append(str(Path.cwd().parent))
from src.data_loader import DataLoader
from src.feature_engineer import FeatureEngineer

warnings.filterwarnings('ignore')

DATA_RAW = Path.cwd().parent / 'data' / 'raw'
DATA_PROCESSED = Path.cwd().parent / 'data' / 'processed'
DATA_FEATURES = Path.cwd().parent / 'data' / 'features'
DATA_FEATURES.mkdir(parents=True, exist_ok=True)


## 1. Load Processed transactions


In [ ]:
loader = DataLoader(DATA_RAW)
df = loader.load_processed(DATA_PROCESSED / "cleaned_retail.csv")
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])


## 2. Generate CLTV Features and Target
To prevent data leakage, we divide transactions:
- **Observation Window**: Dec 2009 – Sep 2011 (approx 21 months) to calculate behavioral customer features.
- **Holdout/Prediction Window**: Sep 2011 – Dec 2011 (90 days) to calculate actual future spend.


In [ ]:
# Initialize FeatureEngineer
fe = FeatureEngineer(df)

# We use 2011-09-01 as the split date (approx 90 days before max date 2011-12-09)
split_date = pd.Timestamp("2011-09-01")
master_table = fe.build_master_feature_table(split_date, prediction_days=90)

# Save master feature matrix
master_table.to_csv(DATA_FEATURES / "customer_features.csv", index=False)
print(f"Saved feature matrix to: {DATA_FEATURES / 'customer_features.csv'}")
master_table.head()


## 3. Verify Master Feature Table
Inspect columns, distributions of target revenue, and non-zero spending ratios.


In [ ]:
print(f"Total customers: {len(master_table):,}")
print("Features created:")
print(list(master_table.columns))
print(f"\nPercentage of returning active customers: {master_table['target_revenue'].gt(0).mean()*100:.2f}%")
